In [ ]:
import numpy as np
import numpy as np
from numpy.linalg import svd
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score

import numpy as np
np.random.seed(0)
import time

In [44]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### Import data

In [45]:
ds_train = pd.read_csv("data/GoogleMap_reviews/reviews_train.csv")
ds_train.drop(["review_id", "text", "word_count"], axis=1, inplace=True)
ds_train.dropna(inplace=True)
ds_train.rename({"cleaned_text": "text"}, axis=1, inplace=True)

In [46]:
ds_val = pd.read_csv("data/GoogleMap_reviews/reviews_val.csv")
ds_val.drop(["review_id", "text", "word_count"], axis=1, inplace=True)
ds_val.dropna(inplace=True)
ds_val.rename({"cleaned_text": "text"}, axis=1, inplace=True)

In [47]:
ds_test = pd.read_csv("data/GoogleMap_reviews/reviews_test.csv")
ds_test.drop(["review_id", "text", "word_count"], axis=1, inplace=True)
ds_test.dropna(inplace=True)
ds_test.rename({"cleaned_text": "text"}, axis=1, inplace=True)

In [48]:
texts = pd.concat([ds_train["text"], ds_val["text"], ds_test["text"]])

### TF-IDF

In [49]:
from joblib import dump

In [50]:
vectorizer = TfidfVectorizer(max_features=10_000, max_df=0.5, ngram_range=(1, 2))
# vectorizer = CountVectorizer(max_features=10_000)
vectorizer.fit(texts)
with open("./models/tfidf-vectorize-10k.pkl", "wb") as f:
    dump(vectorizer, f, protocol=5)

In [51]:
X_train, X_test = vectorizer.transform(ds_train["text"]), vectorizer.transform(ds_test["text"])
y_train, y_test = ds_train["rating"], ds_test["rating"]
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(755517, 10000)
(94434, 10000)
(755517,)
(94434,)


In [52]:
from sklearn.svm import LinearSVC
start_time = time.time()

model = LinearSVC()
model.fit(X_train, y_train)

end_time = time.time()
process_time = round(end_time-start_time,2)
print("Fitting SVC took {} seconds".format(process_time))

Fitting SVC took 75.68 seconds


In [53]:
with open("./models/svm-tfidf-10k.pkl", "wb") as f:
    dump(model, f, protocol=5)

In [56]:
predictions = model.predict(X_test)
print("Accuracy of model is {:.4%}".format(accuracy_score(y_test,predictions)))

Accuracy of model is 72.1054%


In [59]:
from sklearn.metrics import classification_report
print(classification_report(y_test, predictions, digits=4))

              precision    recall  f1-score   support

           1     0.6146    0.7708    0.6839      7154
           2     0.3652    0.0833    0.1356      3902
           3     0.4379    0.2265    0.2986      6900
           4     0.4741    0.2263    0.3064     17197
           5     0.7803    0.9581    0.8601     59281

    accuracy                         0.7211     94434
   macro avg     0.5344    0.4530    0.4569     94434
weighted avg     0.6698    0.7211    0.6749     94434

